In [1]:
!pip uninstall -y edgar edgartools
!pip list | grep edgar
!pip install edgartools --no-cache-dir

Found existing installation: edgartools 5.26.1
Uninstalling edgartools-5.26.1:
  Successfully uninstalled edgartools-5.26.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 21.2 MB/s eta 0:00:00 0:00:01


In [5]:
from google.colab import drive
drive.mount('/content/drive')

from edgar import set_identity, Company
from pathlib import Path
import time

data_dir = Path("/content/drive/MyDrive/data/Sections")
data_dir.mkdir(parents=True, exist_ok=True)

set_identity("Rens Gerritsen 658549rg@eur.nl")

tickers = ["0000001750", "0000002969", "0000002488"]
start_yr = 2025


def find_section(sections, target):
    target = target.lower().replace(" ", "")
    
    for s in sections:
        if isinstance(s, dict):
            item = s.get('item', '').lower().replace(" ", "")
            
            if target in item:
                return s.get('text')
    
    return None


def download_filings():
    for ticker in tickers:
        print(f"\n--- processing {ticker} ---")
        try:
            company = Company(ticker)
            filings = company.get_filings(form="10-K")

            for filing in filings:
                year = filing.filing_date.year

                if year < start_yr:
                    continue
                
                strat_path = data_dir / f"{ticker}_{year}_strategy.txt"
                risk_path = data_dir / f"{ticker}_{year}_risk.txt"
                mgmt_path = data_dir / f"{ticker}_{year}_MD&A.txt"

                if strat_path.exists() and risk_path.exists() and mgmt_path.exists():
                    print(f"Skipping {year} (already exists)")
                    continue

                print(f"Downloading {year} 10-K sections...")

                sections = filing.sections()

                strategy_text = find_section(sections, "item 1")
                risk_text = find_section(sections, "item 1a")
                mgmt_text = find_section(sections, "item 7")

                print(f"{year} - Strategy:", strategy_text is not None)
                print(f"{year} - Risk:", risk_text is not None)
                print(f"{year} - MD&A:", mgmt_text is not None)

                if strategy_text:
                    strat_path.write_text(strategy_text, encoding="utf-8")
                if risk_text:
                    risk_path.write_text(risk_text, encoding="utf-8")
                if mgmt_text:
                    mgmt_path.write_text(mgmt_text, encoding="utf-8")

                if not any([strategy_text, risk_text, mgmt_text]):
                    print("⚠️ Falling back to raw HTML")
                    html = filing.html()
                    
                    raw_path = data_dir / f"{ticker}_{year}_raw.html"
                    raw_path.write_text(html, encoding="utf-8")

                print("Saving to:", data_dir)

                time.sleep(0.2)
        
        except Exception as e:
            print(f"Error with {ticker}: {e}")


download_filings()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- processing 0000001750 ---
2025 - Strategy: False
2025 - Risk: False
2025 - MD&A: False
⚠️ Falling back to raw HTML
Saving to: /content/drive/MyDrive/data/Sections

--- processing 0000002969 ---
2025 - Strategy: False
2025 - Risk: False
2025 - MD&A: False
⚠️ Falling back to raw HTML
Saving to: /content/drive/MyDrive/data/Sections

--- processing 0000002488 ---
2026 - Strategy: False
2026 - Risk: False
2026 - MD&A: False
⚠️ Falling back to raw HTML
Saving to: /content/drive/MyDrive/data/Sections
2026 - Strategy: False
2026 - Risk: False
2026 - MD&A: False
⚠️ Falling back to raw HTML
Saving to: /content/drive/MyDrive/data/Sections
2025 - Strategy: False
2025 - Risk: False
2025 - MD&A: False
⚠️ Falling back to raw HTML
Saving to: /content/drive/MyDrive/data/Sections
